# Arquiva produtos inativos no Odoo

Fluxo:
1. Carrega todos os produtos do Focco com seu SIT
2. Carrega produtos existentes no Odoo (`product.template`)
3. Identifica quais estao com SIT inativo no Focco
4. Arquiva no Odoo os produtos inativos no Focco

In [8]:
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
import xmlrpc.client
import pandas as pd

url = URL.create(
    drivername="postgresql+psycopg2",
    username="tiago.premiano",
    password="Sohome@2027",
    host="10.1.57.244",
    port=5432,
    database="dw_sohome",
)
engine = create_engine(url)

ODOO_URL  = "http://docker.gruposohome.com:8070/"
ODOO_DB   = "odoo"
ODOO_USER = "tiago@sohome.com"
ODOO_PASS = "admin"

common = xmlrpc.client.ServerProxy(f"{ODOO_URL}/xmlrpc/2/common")
uid    = common.authenticate(ODOO_DB, ODOO_USER, ODOO_PASS, {})
if not uid:
    raise Exception("Falha na autenticacao Odoo")
models = xmlrpc.client.ServerProxy(f"{ODOO_URL}/xmlrpc/2/object")
print(f"Odoo: conectado como uid={uid}")

Odoo: conectado como uid=2


## 1. Produtos do Focco com SIT

In [ ]:
df_produtos = pd.read_sql(text("""
    SELECT COD_ITEM, DESC_TECNICA AS PRODUTO, SIT
    FROM TITENS
    WHERE SIT = 0
    ORDER BY COD_ITEM
"""), engine)
engine.dispose()

print(f"Produtos inativos no Focco (SIT=0): {len(df_produtos)}")
print()
print(df_produtos.to_string(index=False))


## 2. Quais desses existem no Odoo (ativos)?

In [10]:
cod_items_inativos = df_produtos["cod_item"].dropna().tolist()

registros = models.execute_kw(
    ODOO_DB, uid, ODOO_PASS,
    "product.template", "search_read",
    [[["default_code", "in", cod_items_inativos], ["active", "=", True]]],
    {"fields": ["id", "name", "default_code"], "limit": 0}
)

df_odoo = pd.DataFrame(registros) if registros else pd.DataFrame(columns=["id", "default_code", "name"])
print(f"Produtos a arquivar no Odoo: {len(df_odoo)}")
if not df_odoo.empty:
    print()
    print(df_odoo[["id", "default_code", "name"]].to_string(index=False))


Produtos a arquivar no Odoo: 0


## 3. Arquiva no Odoo

**Revise a lista acima antes de rodar esta celula.**

In [ ]:
if df_odoo.empty:
    print("Nenhum produto para arquivar.")
else:
    ids_arquivar = df_odoo["id"].tolist()
    BATCH = 200
    ok = err = 0

    for i in range(0, len(ids_arquivar), BATCH):
        sub = ids_arquivar[i:i + BATCH]
        try:
            models.execute_kw(
                ODOO_DB, uid, ODOO_PASS,
                "product.template", "write",
                [sub, {"active": False}]
            )
            ok += len(sub)
        except Exception as e:
            print(f"  Erro lote {i//BATCH}: {e}")
            err += len(sub)

    print(f"Arquivados: {ok} OK | {err} erros")